<a href="https://colab.research.google.com/github/erwanBellon/2025_ML_EES/blob/main/project/code/XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part I: Setup



In [12]:

# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Is this notebook running on Colab or Kaggle?
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

# Cloning repo or fetch latest changes and path management
!git clone https://github.com/erwanBellon/2025_ML_EES.git
%cd /content/2025_ML_EES
!git pull

import os
from pathlib import Path

# Move into the project directory
%cd /content/2025_ML_EES/project/code
print("Current working directory:", Path.cwd())

# Define main project dir and outputs
PROJECT_ROOT_DIR = Path.cwd().parent       # -> /content/2025_ML_EES/project
OUTPUTS_PATH = PROJECT_ROOT_DIR / "outputs"
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to:", OUTPUTS_PATH)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
assert tf.__version__ >= "2.0"

if not tf.config.list_physical_devices('GPU'):
    print("No GPU detected. CNNs can be slow without GPU.")

# Common imports
import pandas as pd
import numpy as np
!pip install rasterio
import rasterio
from tf.keras.models import load_model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


# To make notebook reproducible
np.random.seed(42)
tf.random.set_seed(42)

# For plots
import matplotlib.pyplot as plt
%matplotlib inline

# Load Tensorboard
%load_ext tensorboard

fatal: destination path '2025_ML_EES' already exists and is not an empty directory.
/content/2025_ML_EES
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 661 bytes | 132.00 KiB/s, done.
From https://github.com/erwanBellon/2025_ML_EES
   0ada102..51157d5  main       -> origin/main
Updating 0ada102..51157d5
Fast-forward
 project/code/CNN_model_v3.ipynb | 30 ++++++++++++++++++++----------
 1 file changed, 20 insertions(+), 10 deletions(-)
/content/2025_ML_EES/project/code
Current working directory: /content/2025_ML_EES/project/code
Outputs will be saved to: /content/2025_ML_EES/project/outputs
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [2]:
print(Path.cwd())

/content/2025_ML_EES/project/code


# Part 2: Load files
## 2.1: Load the model using a CNN

In [6]:
# Path to your saved model
model_path = Path.cwd() /"../outputs/cnn_bestModel.keras"

# Load model
model = load_model(model_path)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 30, 30, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip         │ (None, 30, 30, 1) │          0 │ image_input[0][0] │
│ (RandomFlip)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom         │ (None, 30, 30, 1) │          0 │ random_flip[0][0] │
│ (RandomZoom)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_translation  │ (None, 30, 30, 1) │          0 │ random_zoom[0][0] │
│ (RandomTranslation) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 30, 30,    │        320 │ random_translati… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 30, 30,    │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 15, 15,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 15, 15,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 7, 7, 64)  │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ table_input         │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 3136)      │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         48 │ table_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │    100,384 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8)         │        136 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 40)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      1,312 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 8)         │        264 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │          9 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 390,653 (1.49 MB)

 Trainable params: 130,217 (508.66 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 260,436 (1017.33 KB)

### 2.2 Load image data (and table input)


In [31]:
# --- Load images ---
presences_path = Path("../data/cropped_landcover/presences")
absences_path = Path("../data/cropped_landcover/absences")

def load_images_from_folder(folder):
    tif_files = list(folder.glob("*.tif"))
    images = []
    image_indices = []
    for tif in tif_files:
        with rasterio.open(tif) as src:
            img = src.read()
            img = np.transpose(img, (1,2,0))
            images.append(img.astype(np.float32))
        # Extract the line index from the filename (assuming `crop_3000_118.tif`)
        idx = int(tif.stem.split("_")[-1])
        image_indices.append(idx)
    return np.array(images), np.array(image_indices)

images_pres, indices_pres = load_images_from_folder(presences_path)
images_abs, indices_abs = load_images_from_folder(absences_path)
print(f"Presences: {images_pres.shape}, Absences: {images_abs.shape}")

# Build dataset & labels
X = np.concatenate([images_pres, images_abs], axis=0)
y = np.concatenate([np.ones(len(images_pres)), np.zeros(len(images_abs))], axis=0).astype(np.int32)
image_indices = np.concatenate([indices_pres, indices_abs], axis=0)  # all image row indices

# --- Load table data ---
rds_path = Path("../data/Table_preds/function_3_100.rds")
!pip install pyreadr
import pyreadr
result = pyreadr.read_r(rds_path)
table_df = result[None]  # get DataFrame

# Select only the rows corresponding to actual images
table_features_all = table_df[['MAP','MAT']].astype(float)
table_features = table_features_all.iloc[image_indices].reset_index(drop=True)

# Normalize
table_features = (table_features - table_features.min()) / (table_features.max() - table_features.min())
table_features = table_features.to_numpy(dtype=np.float32)

X_train, X_temp, table_train, table_temp, y_train, y_temp = train_test_split(
    X, table_features, y, test_size=0.2, random_state=42, stratify=y
)
X_valid, X_test, table_valid, table_test, y_valid, y_test = train_test_split(
    X_temp, table_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)



Presences: (117, 30, 30, 1), Absences: (200, 30, 30, 1)


In [32]:
train_ds = tf.data.Dataset.from_tensor_slices(((X_train, table_train), y_train))
valid_ds = tf.data.Dataset.from_tensor_slices(((X_valid, table_valid), y_valid))
test_ds  = tf.data.Dataset.from_tensor_slices(((X_test, table_test), y_test))


# Add `.name` attribute like TFDS
train_ds.name = "Training"
valid_ds.name = "Validation"
test_ds.name  = "Test"

In [41]:
def preprocess_multi_inputs(inputs, label):
    image, table = inputs
    # Convert image to float32 and add channel dimension if needed
    image = tf.cast(image, tf.float32)
    # Replace Nan (non-forest) with 0s
    image = tf.where(tf.math.is_nan(image), 0.0, image)

    if len(image.shape) == 3:  # (H,W,C) or (H,W)
        image = tf.expand_dims(image, -1)  # ensures (H,W,1)
    # Table should already be (batch_size, 2) after batching
    table = tf.cast(table, tf.float32)
    label = tf.cast(label, tf.float32)
    return (image, table), label

In [42]:
train_ds = (
    tf.data.Dataset.from_tensor_slices(((X_train, table_train), y_train))
      .shuffle(1000, reshuffle_each_iteration=True)   # <--- SHUFFLE HERE
      .map(preprocess_multi_inputs, num_parallel_calls=tf.data.AUTOTUNE)
      .prefetch(tf.data.AUTOTUNE)
)
valid_ds = (
    tf.data.Dataset.from_tensor_slices(((X_valid, table_valid), y_valid))
      .map(preprocess_multi_inputs, num_parallel_calls=tf.data.AUTOTUNE)
      .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices(((X_test, table_test), y_test))
      .map(preprocess_multi_inputs, num_parallel_calls=tf.data.AUTOTUNE)
      .prefetch(tf.data.AUTOTUNE)
)

# Add `.name` attribute like TFDS
train_ds.name = "Training"
valid_ds.name = "Validation"
test_ds.name  = "Test"


Unbatch the test dataset and prepare 5 sample images


In [46]:
sample_images = []
sample_tables = []

for (img, tab), label in test_ds.take(5):  # unbatch to get single samples
    # Image preprocessing
    img = tf.cast(img, tf.float32)
    img = tf.where(tf.math.is_nan(img), 0.0, img)
    if len(img.shape) == 2:  # (H,W)
        img = tf.expand_dims(img, -1)  # (H,W,1)
    sample_images.append(img.numpy())

    # Table preprocessing
    tab = tf.cast(tab, tf.float32)
    sample_tables.append(tab.numpy())

# Convert to numpy arrays with batch dimension
sample_images = np.stack(sample_images)
sample_tables = np.stack(sample_tables)

print("Sample images shape:", sample_images.shape)   # (5,H,W,C)
print("Sample tables shape:", sample_tables.shape)   # (5,n_features)
print("Min/Max values:", img.numpy().min(), img.numpy().max())



Sample images shape: (5, 30, 30, 1, 1)
Sample tables shape: (5, 2)
Min/Max values: 0.0 1.0


## PART 3. Looking at the feature maps

In [8]:
# Print all layers with their output shapes
for i, layer in enumerate(model.layers):
    try:
        shape = layer.output.shape
    except AttributeError:
        shape = "N/A"  # For InputLayer or unusual layers
    print(i, layer.name, shape, type(layer))


0 image_input (None, 30, 30, 1) <class 'keras.src.layers.core.input_layer.InputLayer'>
1 random_flip (None, 30, 30, 1) <class 'keras.src.layers.preprocessing.image_preprocessing.random_flip.RandomFlip'>
2 random_zoom (None, 30, 30, 1) <class 'keras.src.layers.preprocessing.image_preprocessing.random_zoom.RandomZoom'>
3 random_translation (None, 30, 30, 1) <class 'keras.src.layers.preprocessing.image_preprocessing.random_translation.RandomTranslation'>
4 conv2d (None, 30, 30, 32) <class 'keras.src.layers.convolutional.conv2d.Conv2D'>
5 conv2d_1 (None, 30, 30, 32) <class 'keras.src.layers.convolutional.conv2d.Conv2D'>
6 max_pooling2d (None, 15, 15, 32) <class 'keras.src.layers.pooling.max_pooling2d.MaxPooling2D'>
7 conv2d_2 (None, 15, 15, 64) <class 'keras.src.layers.convolutional.conv2d.Conv2D'>
8 max_pooling2d_1 (None, 7, 7, 64) <class 'keras.src.layers.pooling.max_pooling2d.MaxPooling2D'>
9 table_input (None, 2) <class 'keras.src.layers.core.input_layer.InputLayer'>
10 flatten (None, 

In [53]:


# Get the conv layer of interest
conv_layer = model.get_layer("conv2d_2")
print("conv layer:",conv_layer.name, conv_layer.output.shape)

# Build sub-model from original model ---
# Input = both image and table, output = last conv layer
feature_map_model = Model(inputs=model.inputs, outputs=conv_layer.output)

# Prepare 5 samples from test dataset ---
sample_images = []
sample_tables = []

for (img, tab), label in test_ds.take(5):  # unbatch to get single samples
    # Image preprocessing
    img = tf.cast(img, tf.float32)
    img = tf.where(tf.math.is_nan(img), 0.0, img)
    if len(img.shape) == 2:  # (H,W)
        img = tf.expand_dims(img, -1)  # (H,W,1)
    sample_images.append(img.numpy())

    # Table preprocessing
    tab = tf.cast(tab, tf.float32)
    sample_tables.append(tab.numpy())

# Convert to numpy arrays with batch dimension
sample_images = np.stack(sample_images)
sample_tables = np.stack(sample_tables)

print("Sample images shape:", sample_images.shape)   # (5,H,W,C)
print("Sample tables shape:", sample_tables.shape)   # (5,n_features)
print("Min/Max values:", img.numpy().min(), img.numpy().max())




conv layer: conv2d_2 (None, 15, 15, 64)
Sample images shape: (5, 30, 30, 1, 1)
Sample tables shape: (5, 2)
Min/Max values: 0.0 1.0


In [61]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Custom colormap: 0 = white, 1 = dark green
cmap_img = ListedColormap(['white', 'darkgreen'])

# Map label to text
label_map = {1: "Presence", 0: "Absence"}

# Directory to save plots in Colab
save_dir = Path("/content/outputs")
save_dir.mkdir(parents=True, exist_ok=True)

for i in range(len(sample_images)):
    img_batch = sample_images[i:i+1]   # shape (1,H,W,C)
    tab_batch = sample_tables[i:i+1]   # shape (1,n_features)
    label = y_test[i]                   # label for the sample

    # Predict feature maps
    feature_maps = feature_map_model.predict([img_batch, tab_batch])

    # Compute mean activation per filter
    mean_activation = feature_maps.mean(axis=(1,2))  # shape (1, n_filters)

    # Get indices of top 3 most activated filters
    top3_idx = np.argsort(mean_activation[0])[::-1][:3]

    # Plot: original + 3 feature maps
    fig, axes = plt.subplots(1, 4, figsize=(16,5))

    # Original image with dark green for 1
    axes[0].imshow(img_batch[0, :, :, 0], cmap=cmap_img, vmin=0, vmax=1)
    axes[0].set_title(f"Original Image\n{label_map[label]}")
    axes[0].axis("off")

    # Feature maps
    for j, idx in enumerate(top3_idx):
        activation = feature_maps[0, :, :, idx]
        # Normalize activation to [0,1] for visualization
        activation = (activation - activation.min()) / (activation.max() - activation.min() + 1e-8)

        im = axes[j+1].imshow(activation, cmap="viridis")
        axes[j+1].set_title(f"Filter {idx}")
        axes[j+1].axis("off")
        fig.colorbar(im, ax=axes[j+1], fraction=0.046, pad=0.04)

    # Save figure to Colab
    save_path = save_dir / f"sample_{i}_{label_map[label]}.png"
    plt.savefig(save_path, bbox_inches="tight")
    plt.close(fig)  # Free memory

print(f"Plots saved to {save_dir}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Plots saved to /content/outputs
